# Phase 1: Data Loading & Inspection

### Step 1: Data Loading

In [34]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Load the data 
# Using "../" goes up one directory level to find the "data" folder
file_path = "../data/risk_factors_cervical_cancer.csv"

df = pd.read_csv(file_path)

# 2. Check the shape
rows, cols = df.shape
print(f"Dataset Shape: {rows} patients and {cols} features\n")

# Peek at the first few rows
display(df.head())

Dataset Shape: 858 patients and 36 features



,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
2,34,1.0,?,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,?,?,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,?,?,0,0,0,0,0,0,0,0


### Step 2: Initial Diagnostic Checks

In [35]:
# 3. Inspect Data Types
print("--- DataFrame Info ---")
df.info()
print("\n")

# 4. Investigate Suspicious 'Object' Columns
suspicious_col = 'Number of sexual partners' 
print(f"--- Unique values in '{suspicious_col}' ---")
print(df[suspicious_col].unique()) 
print("\n")

# 5. Identify the Target Variable
target = 'Biopsy'
print(f"--- Target Variable Distribution ({target}) ---")
display(df[target].value_counts(normalize=True).round(4) * 100)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Age                                 858 non-null    int64 
 1   Number of sexual partners           858 non-null    object
 2   First sexual intercourse            858 non-null    object
 3   Num of pregnancies                  858 non-null    object
 4   Smokes                              858 non-null    object
 5   Smokes (years)                      858 non-null    object
 6   Smokes (packs/year)                 858 non-null    object
 7   Hormonal Contraceptives             858 non-null    object
 8   Hormonal Contraceptives (years)     858 non-null    object
 9   IUD                                 858 non-null    object
 10  IUD (years)                         858 non-null    object
 11  STDs                               

Biopsy
0    93.59
1     6.41
Name: proportion, dtype: float64

# Phase 2: Data Cleaning & Type Conversion

In [36]:
# 1. Replace the text '?' with standard numpy NaN
df = df.replace('?', np.nan)

# 2. Convert all columns to numeric (Future-Proof Version)
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError:
        pass

print("Data types successfully converted!\n")

print("--- Updated DataFrame Info ---")
df.info()

Data types successfully converted!

--- Updated DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 858 non-null    int64  
 1   Number of sexual partners           832 non-null    float64
 2   First sexual intercourse            851 non-null    float64
 3   Num of pregnancies                  802 non-null    float64
 4   Smokes                              845 non-null    float64
 5   Smokes (years)                      845 non-null    float64
 6   Smokes (packs/year)                 845 non-null    float64
 7   Hormonal Contraceptives             750 non-null    float64
 8   Hormonal Contraceptives (years)     750 non-null    float64
 9   IUD                                 741 non-null    float64
 10  IUD (years)                         741 non

# Phase 3: Data Assessment & Investigation

### Step 1: Consistency check

In [37]:
print("==========================================")
print("PHASE 3: DATA ASSESSMENT & INVESTIGATION")
print("==========================================")
print("--- Step 1: Logical Consistency Checks ---\n")

# ---------------------------------------------------------
# TEST 1: The "Smokes" Logic
# If a patient has > 0 years smoking or > 0 packs/year, 
# their master 'Smokes' column MUST be 1.0 (True).
# ---------------------------------------------------------
print("TEST 1: 'Smokes' Consistency")
# Find rows where Smokes is 0 or NaN, BUT they have smoking years or packs recorded
smokes_flaws = df[
    ((df['Smokes'] == 0) | (df['Smokes'].isnull())) & 
    ((df['Smokes (years)'] > 0) | (df['Smokes (packs/year)'] > 0))
]

print(f"Found {len(smokes_flaws)} patients with contradictory 'Smokes' data.")
if len(smokes_flaws) > 0:
    print("Example of the flawed data:")
    print(smokes_flaws[['Smokes', 'Smokes (years)', 'Smokes (packs/year)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 2: The "STDs" Logic (Updated)
# If a patient has ANY data > 0 in ANY column starting with 'STDs', 
# their master 'STDs' column MUST be 1.0 (True).
# ---------------------------------------------------------
print("TEST 2: 'STDs' Consistency")

# Grab ALL columns that start with 'STDs' EXCEPT the master 'STDs' column itself
all_std_related_cols = [col for col in df.columns if col.startswith('STDs') and col != 'STDs']

# Find rows where master STDs is 0 or NaN, BUT the sum of any related STD column is > 0
# .sum(axis=1) safely ignores NaNs and adds up all the numbers in that patient's row
stds_flaws = df[
    ((df['STDs'] == 0) | (df['STDs'].isnull())) & 
    (df[all_std_related_cols].sum(axis=1) > 0)
]

print(f"Found {len(stds_flaws)} patients with contradictory 'STDs' data.")
if len(stds_flaws) > 0:
    print("Example of the flawed data:")
    # We will print the master column and the first few STD columns to keep the terminal clean
    cols_to_show = ['STDs'] + all_std_related_cols[:4]
    print(stds_flaws[cols_to_show].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 3: The "Reverse" Logic
# If a patient's master column says they DO NOT smoke/have STDs (0), 
# then their sub-columns MUST be 0.
# ---------------------------------------------------------
print("TEST 3: Reverse Logic (0s should mean 0s)")
# Check Smokes
smokes_reverse_flaws = df[(df['Smokes'] == 0) & ((df['Smokes (years)'].isnull()) | (df['Smokes (packs/year)'].isnull()))]
print(f"Found {len(smokes_reverse_flaws)} patients who are non-smokers (0) but have NaN in years/packs.")

# Check STDs
stds_reverse_flaws = df[(df['STDs'] == 0) & (df['STDs (number)'].isnull())]
print(f"Found {len(stds_reverse_flaws)} patients who have NO STDs (0) but have NaN in STDs (number).")

PHASE 3: DATA ASSESSMENT & INVESTIGATION
--- Step 1: Logical Consistency Checks ---

TEST 1: 'Smokes' Consistency
Found 0 patients with contradictory 'Smokes' data.
----------------------------------------

TEST 2: 'STDs' Consistency
Found 0 patients with contradictory 'STDs' data.
----------------------------------------

TEST 3: Reverse Logic (0s should mean 0s)
Found 0 patients who are non-smokers (0) but have NaN in years/packs.
Found 0 patients who have NO STDs (0) but have NaN in STDs (number).


### Step 2: Missing data handling

In [38]:
print("--- Step 2: Missing Data Assessment ---")

# Calculate the percentage of missing values per column
# df.isnull().sum() counts the NaNs, dividing by len(df) gets the ratio, * 100 gets percentage
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Filter to only show columns that have AT LEAST SOME missing data (> 0%)
missing_cols = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

print(f"Total columns with missing data: {len(missing_cols)} out of {df.shape[1]}\n")

if len(missing_cols) > 0:
    print("--- SEVERE Missing Data (> 50%) [Drop Candidates] ---")
    severe_cols = missing_cols[missing_cols > 50]
    if len(severe_cols) > 0:
        print(severe_cols.round(2).astype(str) + ' %')
    else:
        print("None")
        
    print("\n--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---")
    moderate_cols = missing_cols[missing_cols <= 50]
    if len(moderate_cols) > 0:
        print(moderate_cols.round(2).astype(str) + ' %')
    else:
        print("None")
print("\n" + "="*40 + "\n")

--- Step 2: Missing Data Assessment ---
Total columns with missing data: 26 out of 36

--- SEVERE Missing Data (> 50%) [Drop Candidates] ---
STDs: Time since first diagnosis    91.72 %
STDs: Time since last diagnosis     91.72 %
dtype: object

--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---
IUD (years)                           13.64 %
IUD                                   13.64 %
Hormonal Contraceptives               12.59 %
Hormonal Contraceptives (years)       12.59 %
STDs (number)                         12.24 %
STDs                                  12.24 %
STDs:vulvo-perineal condylomatosis    12.24 %
STDs:vaginal condylomatosis           12.24 %
STDs:pelvic inflammatory disease      12.24 %
STDs:syphilis                         12.24 %
STDs:genital herpes                   12.24 %
STDs:molluscum contagiosum            12.24 %
STDs:HIV                              12.24 %
STDs:AIDS                             12.24 %
STDs:condylomatosis                   12.24

# Phase 4: Handling Missing Data (Imputation)

In [39]:
print("==========================================")
print("PHASE 4: HANDLING MISSING DATA (IMPUTATION)")
print("==========================================")

# ---------------------------------------------------------
# 1. Drop the "Not Applicable" Columns
# ---------------------------------------------------------
cols_to_drop = ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped {len(cols_to_drop)} columns due to >90% missing data.\n")

# ---------------------------------------------------------
# 2. Impute STDs with the Median
# ---------------------------------------------------------
print("--- STD Columns Imputation Log ---")
std_cols = [col for col in df.columns if col.startswith('STDs')]

imputed_count = 0
for col in std_cols:
    # Only calculate and print if the column actually has missing data
    if df[col].isnull().sum() > 0:
        col_median = df[col].median()
        df[col] = df[col].fillna(col_median)
        print(f"'{col}' missing values filled with its median: {col_median}")
        imputed_count += 1

print(f"Finished imputing {imputed_count} STD-related columns.\n")

# ---------------------------------------------------------
# 3. Conditional Imputation (Contraceptives & Smokes)
# ---------------------------------------------------------
print("--- Conditional Imputation Log ---")
# We structure this as a list of (Master Column, [List of Sub-Columns])
conditional_groups = [
    ('IUD', ['IUD (years)']), 
    ('Hormonal Contraceptives', ['Hormonal Contraceptives (years)']),
    ('Smokes', ['Smokes (years)', 'Smokes (packs/year)'])
]

for master, sub_cols in conditional_groups:
    # Step A: Fill the master categorical column with its median (0 or 1)
    master_median = df[master].median()
    df[master] = df[master].fillna(master_median)
    print(f"Master '{master}' missing values filled with: {master_median}")
    
    for sub in sub_cols:
        # Step B: Calculate the median strictly for patients who used it (> 0)
        user_median = df[df[master] == 1][sub].median()
        
        # Step C: Apply conditional logic to fill the missing sub-columns
        df.loc[df[master] == 0, sub] = df.loc[df[master] == 0, sub].fillna(0)
        df.loc[df[master] == 1, sub] = df.loc[df[master] == 1, sub].fillna(user_median)
        
        print(f"  -> '{sub}' missing values conditionally filled:")
        print(f"     - With 0.0 if '{master}' == 0")
        print(f"     - With {user_median} if '{master}' == 1")
print("\n")

# ---------------------------------------------------------
# 4. Global Fallback for remaining columns
# ---------------------------------------------------------
print("--- Standard Imputation Log (Global Fallback) ---")
remaining_imputed = 0
for col in df.columns:
    if df[col].isnull().sum() > 0:
        col_median = df[col].median()
        df[col] = df[col].fillna(col_median)
        print(f"'{col}' missing values filled with: {col_median}")
        remaining_imputed += 1

if remaining_imputed == 0:
    print("No remaining columns needed fallback imputation.")

# ---------------------------------------------------------
# 5. Final Proof
# ---------------------------------------------------------
print("\n--- Final Dataset Check ---")
print(f"Total missing values left in the ENTIRE dataset: {df.isnull().sum().sum()}")
print(f"Final Dataset Shape: {df.shape[0]} patients, {df.shape[1]} features")
print("==========================================\n")

PHASE 4: HANDLING MISSING DATA (IMPUTATION)
Dropped 2 columns due to >90% missing data.

--- STD Columns Imputation Log ---
'STDs' missing values filled with its median: 0.0
'STDs (number)' missing values filled with its median: 0.0
'STDs:condylomatosis' missing values filled with its median: 0.0
'STDs:cervical condylomatosis' missing values filled with its median: 0.0
'STDs:vaginal condylomatosis' missing values filled with its median: 0.0
'STDs:vulvo-perineal condylomatosis' missing values filled with its median: 0.0
'STDs:syphilis' missing values filled with its median: 0.0
'STDs:pelvic inflammatory disease' missing values filled with its median: 0.0
'STDs:genital herpes' missing values filled with its median: 0.0
'STDs:molluscum contagiosum' missing values filled with its median: 0.0
'STDs:AIDS' missing values filled with its median: 0.0
'STDs:HIV' missing values filled with its median: 0.0
'STDs:Hepatitis B' missing values filled with its median: 0.0
'STDs:HPV' missing values fill

# Phase 5: Export clean data

In [41]:
# Phase 5: Export Clean Data
# Save the cleaned dataset so we can pick it up in the next notebook
output_path = "../data/clean_cervical_cancer_data.csv"
df.to_csv(output_path, index=False)

print(f"Clean data successfully saved to {output_path}")

Clean data successfully saved to ../data/clean_cervical_cancer_data.csv
